# Sentiment Analysis

**Objective:** Build a machine learning model that classifies the sentiment of text data
(positive, negative, or neutral) to gain insight into public opinion or customer feedback.

**Dataset:** Download a sentiment dataset from Kaggle — good options: "Twitter Sentiment
Analysis", "Amazon Product Reviews", or "IMDB Movie Reviews". Save it as `reviews.csv`
in the same folder as this notebook. This notebook assumes two columns: `text` (the review/tweet)
and `sentiment` (label). Rename your dataset's columns to match, or adjust the column
names in Section 2 below.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

sns.set_style('whitegrid')
%matplotlib inline

## 2. Load Data & Inspect Class Distribution

In [ ]:
df = pd.read_csv('reviews.csv')

# Rename these to match your dataset's actual column names
TEXT_COL = 'text'
LABEL_COL = 'sentiment'

df = df[[TEXT_COL, LABEL_COL]].dropna()
print(df.shape)
df[LABEL_COL].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=LABEL_COL, data=df)
plt.title('Sentiment Class Distribution')
plt.show()

## 3. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)                 # remove punctuation/numbers
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]  # remove stopwords
    return ' '.join(tokens)

df['clean_text'] = df[TEXT_COL].apply(clean_text)
df[[TEXT_COL, 'clean_text']].head()

## 4. Feature Extraction: TF-IDF

**Purpose of TF-IDF:** Term Frequency-Inverse Document Frequency converts text into
numeric vectors that weigh words by how often they appear in a document, offset by how
common they are across all documents. This down-weights generic words (like "good" if
it appears everywhere) and highlights words that are more distinctive to a particular
review, improving the classifier's ability to separate sentiment classes.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['clean_text'])
y = df[LABEL_COL]

print(X.shape)

## 5. Train/Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## 6. Train Classifiers: Naive Bayes + SVM

In [ ]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

svm = LinearSVC(random_state=42)
svm.fit(X_train, y_train)

models = {'Naive Bayes': nb, 'Linear SVM': svm}

## 7. Evaluation

In [ ]:
results = {}
for name, model in models.items():
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test, preds, average='weighted', zero_division=0)
    f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
    results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

    print(f"===== {name} =====")
    print(classification_report(y_test, preds, zero_division=0))
    print()

pd.DataFrame(results).T

## 8. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
labels = sorted(y.unique())

for ax, (name, model) in zip(axes, models.items()):
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 9. Sentiment Distribution & WordClouds

In [ ]:
# pip install wordcloud --break-system-packages   (run once in terminal if not installed)
from wordcloud import WordCloud

fig, axes = plt.subplots(1, len(labels), figsize=(6*len(labels), 5))
if len(labels) == 1:
    axes = [axes]

for ax, label in zip(axes, labels):
    text = ' '.join(df[df[LABEL_COL] == label]['clean_text'])
    wc = WordCloud(width=500, height=400, background_color='white').generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'WordCloud: {label}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 10. Error Analysis

In [ ]:
best_model_name = max(results, key=lambda k: results[k]['f1'])
best_model = models[best_model_name]
print(f"Analyzing errors for: {best_model_name}")

test_idx = y_test.index
preds = best_model.predict(X_test)
mismatch_mask = preds != y_test.values

mismatches = pd.DataFrame({
    'text': df.loc[test_idx, TEXT_COL].values[mismatch_mask],
    'actual': y_test.values[mismatch_mask],
    'predicted': preds[mismatch_mask]
})

mismatches.head(5)

**Discuss here:** For each of the 5 misclassified examples above, note any patterns —
e.g. sarcasm, mixed sentiment in one review, negation words ("not good") confusing the
model, or very short/ambiguous text.

## Conclusion

Write 2-3 sentences here on:
- Which model performed best (accuracy/F1) and by how much
- What real-world application this could serve (e.g. monitoring product reviews,
  social media brand sentiment, customer support triage)